# Day 55 — Code execution agents

`CodeExecutorAgent` runs code blocks that another agent writes. **Security:** this executes
real code on your machine — in production use a Docker/sandboxed executor and never run
untrusted code.

> Install: `pip install "autogen-agentchat" "autogen-ext[openai]"` and set `OPENAI_API_KEY`. For a local model, pass `base_url=` + `model_info=` to the client. Also needs a local executor from `autogen_ext.code_executors`.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. Solution below.

In [ ]:
from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

mc = OpenAIChatCompletionClient(model="gpt-4o-mini")
coder = AssistantAgent("coder", model_client=mc,
    system_message="Write a SHORT Python code block that prints the answer, then stop.")
executor = CodeExecutorAgent("executor", code_executor=LocalCommandLineCodeExecutor(work_dir="coding"))

# TODO: team = RoundRobinGroupChat([coder, executor], termination_condition=MaxMessageTermination(4))
#       result = await team.run(task="Print the sum of 1..10 with Python."); print last message; close mc


## 🔒 Solution

Verified against autogen-agentchat 0.7.5.

In [ ]:
from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

mc = OpenAIChatCompletionClient(model="gpt-4o-mini")
coder = AssistantAgent("coder", model_client=mc,
    system_message="Write a SHORT Python code block that prints the answer, then stop.")
executor = CodeExecutorAgent("executor", code_executor=LocalCommandLineCodeExecutor(work_dir="coding"))

team = RoundRobinGroupChat([coder, executor], termination_condition=MaxMessageTermination(4))
result = await team.run(task="Print the sum of 1..10 with Python.")
print(result.messages[-1].content)
await mc.close()